# Library

In [74]:
!pip install Sastrawi

In [70]:
import pandas as pd
import string
import re
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.linear_model import LogisticRegression

# Data

In [71]:
data_path = '/content/drive/MyDrive/UTY/Semester 7/Pemrosesan Teks /Klasifikasi Teks/data_labeled_mlbb.csv'

# Preprocessing

In [72]:
df = pd.read_csv(data_path)

# Lowercase
df['clean'] = df['content'].str.lower()

# Remove Punctuation
def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

df['clean'] = df['clean'].apply(remove_punctuation)

# Remove Repeating Characters
def remove_repeating_chars(text):
    return re.sub(r'(.)\1{2,}', r'\1\1', text)

df['clean'] = df['clean'].apply(remove_repeating_chars)

# Remove Short Words
def remove_short_words(text, min_len=3):
    words = text.split()
    long_words = [word for word in words if len(word) >= min_len]
    return ' '.join(long_words)

df['clean'] = df['clean'].apply(remove_short_words)

df.head()

,userName,content,label,clean
0,JIDAPOWER JIDA,malas nak cakap lagi dh enemy dh la pro tim dh...,negative,malas nak cakap lagi enemy pro tim noob lagi p...
1,Galih Najid,game ytim,negative,game ytim
2,Muhammad Nibras,bagus saya suka tapi berikan saya ws lebih ban...,positive,bagus saya suka tapi berikan saya lebih banyak...
3,Ken Chuu,udah semenjak ganti season baru ini . makin ng...,negative,udah semenjak ganti season baru ini makin ngac...
4,IND khadafi,game nya udah bagus tapi. tolong perbaikilah m...,neutral,game nya udah bagus tapi tolong perbaikilah mu...


## Replace Slang Words

In [73]:
slang_dict = {
      'bagu':'bagus',
      'bgt':'banget',
      'gak':'tidak',
      'gk':'tidak',
      'gw':'aku',
      'trus':'terus',
      'monton':'moonton',
      'montoon':'moonton',
      'maen':'main',
      'leg':'lag',
      'ngelag':'lag',
      'ngeleg':'lag',
      'klo':'kalau',
      'tim':'team',
      'ga':'tidak',
      'lu':'kamu',
      'gem':'game',
      'gua':'aku',
      'saya':'aku',
      'g':'tidak',
      'kalo':'kalau',
      'gin':'game',
      'yg':'yang',
      'ngelek':'lag',
      'sering lag':'lag',
      'kayak':'seperti',
      'kaya':'seperti',
      'geme':'game',
      'i':'aku',
      'ny':'nya',
      'gamenya':'game',
      'x':'kali',
      'dark':'gelap',
      'bener':'benar',
      'benerin':'rusak',
      'dibenerin':'rusak',
      'beneran':'serius',
      'gabener':'salah',
      'benerr':'benar',
      'benerlah':'benar',
      'bbenerin':'rusak',
      'benere':'benar',
      'benerlaa':'benar',
      'ngasih':'beri',
      'udh':'sudah',
      'ni':'ini',
      'karna':'karena',
      'min':'minimal',
      'burik':'buruk',
      'sampe':'sampai',
      'pake':'pakai',
      'drak':'gelap',
      'emang':'memang',
      'dikit':'sedikit',
      'dikitlh':'sedikit',
      'cuman':'hanya',
      'dah':'sudah',
      'berik':'beri',
      'ajha':'aja',
      'kek':'seperti',
      'lapor':'report',
      'laporin':'report',
      'ngelaporin':'report',
      'reportnya':'report',
      'direport':'report',
      'ngreport':'report',
      'cobain':'coba',
      'coban':'coba',
      'temen':'teman',
      'temenya':'teman',
      'temanteman':'teman',
      'temantemanku':'teman',
      'jagoteman':'jago',
      'doang':'hanya',
      'drag':'gelap',
      'system':'sistem',
      'dpt':'dapat',
      'tetep':'tetap',
      'stress':'stres',
      'nggak':'tidak',
      'skil':'skill',
      'skillnya':'skill',
      'skills':'skill',
      'bosok':'buruk',
      'bobrok':'buruk',
      'anjlok':'buruk',
      'toksik':'buruk',
      'kalok':'kalau',
      'ok':'oke',
      'okey':'oke',
      'gue':'aku',
      'baguss':'bagus',
      'anda':'kamu',
      'terimakasi':'terimakasih',
      'terima':'terimakasih',
      'makasihh':'terimakasih',
      'trjmakasih':'terimakasih',
      'trimakasih':'terimakasih',
      'makasih':'terimakasih',
      'tpi':'tapi',
      'aj':'aja',
      'jdi':'jadi',
      'ampas':'buruk',
      'woi':'buruk',
      'tp':'tapi',
      'matchmakingnya':'matchmaking',
      'bet':'banget',
      'ytim':'yatim',
      'sangatt':'sangat',
      'inisangat':'sangat'
}

def replace_slang(text):
    words = text.split()
    new_words = [slang_dict.get(word, word) for word in words]
    return ' '.join(new_words)

df['clean'] = df['clean'].apply(replace_slang)

df.head()

,userName,content,label,clean
0,JIDAPOWER JIDA,malas nak cakap lagi dh enemy dh la pro tim dh...,negative,malas nak cakap lagi enemy pro team noob lagi ...
1,Galih Najid,game ytim,negative,game yatim
2,Muhammad Nibras,bagus saya suka tapi berikan saya ws lebih ban...,positive,bagus aku suka tapi berikan aku lebih banyak lagi
3,Ken Chuu,udah semenjak ganti season baru ini . makin ng...,negative,udah semenjak ganti season baru ini makin ngac...
4,IND khadafi,game nya udah bagus tapi. tolong perbaikilah m...,neutral,game nya udah bagus tapi tolong perbaikilah mu...


## Remove Stopwords

In [75]:
factory = StemmerFactory()
stemmer = factory.create_stemmer()

stop_words = set([
    'yang', 'nya', 'dan', 'di', 'ke', 'dari', 'untuk', 'dengan', 'pada', 'adalah', 'itu', 'aja', 'tapi', 'sih', 'dapet', 'tiap', 'saat', 'tapi', 'kasih', 'jaring',
    'pas', 'mulu', 'dulu', 'ini', 'saya', 'kami', 'kita', 'anda', 'mereka', 'atau', 'juga', 'tidak', 'eh', 'nyeselin', 'yah', 'ya', 'nah', 'kok', 'lagi', 'udah',
    'jadi', 'karena', 'agar', 'supaya', 'moonton', 'sebagai', 'oleh', 'terhadap', 'tentang', 'bahwa', 'hanya', 'saja', 'masih', 'telah', 'akan', 'bisa', 'dapat',
    'harus', 'boleh', 'mau', 'ingin', 'perlu', 'nya','game', 'hp', 'gamenya', 'tara', 'ml', 'hok', 'ada', 'aku', 'main', 'hero', 'rank', 'team', 'kalau', 'apa',
    'player', 'terus', 'sama', 'lah', 'nge', 'matchmaking', 'match', 'malah', 'download', 'orang', 'padahal', 'making', 'tau', 'plis', 'tencent', 'malah', 'buat',
    'udah', 'seperti', 'bikin', 'sering', 'kali', 'tiap', 'tiba', 'sekali', 'solo', 'data', 'kok', 'ku', 'saat', 'honor', 'of', 'kings', 'legend', 'mobile', 'dong',
    'belah', 'hari', 'tuh', 'kan', 'in', 'mlbb', 'soal', 'pun', 'login', 'waktu', 'kata', 'moba', 'mode', 'pakai', 'the', 'kalian', 'lane', 'sistem', 'nih', 'war',
    'simpan', 'fitur', 'ketika', 'awal', 'pick', 'cara', 'ama', 'gilir', 'nama', 'map', 'to', 'masak', 'gameplay', 'beli', 'loh', 'kesini', 'sini', 'apalagi', 'turut',
    'deh', 'memang', 'kembali', 'baca', 'bakal', 'temu', 'daripada', 'rasa', 'is', 'a', 'si', 's', 'jam', 'tahun', 'liat', 'depan', 'la', 'sangat', 'kamu'
])

def stem_text(text):
    words = text.split()
    filtered_words = [word for word in words if word not in stop_words]
    stemmed_words = [stemmer.stem(word) for word in filtered_words]
    return ' '.join(stemmed_words)

df['clean'] = df['clean'].apply(stem_text)

df.head()

,userName,content,label,clean
0,JIDAPOWER JIDA,malas nak cakap lagi dh enemy dh la pro tim dh...,negative,malas nak cakap enemy pro noob paling malas re...
1,Galih Najid,game ytim,negative,yatim
2,Muhammad Nibras,bagus saya suka tapi berikan saya ws lebih ban...,positive,bagus suka ikan lebih banyak
3,Ken Chuu,udah semenjak ganti season baru ini . makin ng...,negative,semenjak ganti season baru makin ngaco matchin...
4,IND khadafi,game nya udah bagus tapi. tolong perbaikilah m...,neutral,bagus tolong baik muntun renk computer musuh s...


# Vectorizer

## Tf-Idf

In [76]:
df_tf = df.copy()

In [77]:
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(df_tf['clean'])
feature_names = tfidf_vectorizer.get_feature_names_out()
df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names)


## Bag of Words

In [78]:
df_bow = df.copy()

In [79]:
bow_vectorizer = CountVectorizer()
bow_matrix = bow_vectorizer.fit_transform(df_bow['clean'])
feature_names_bow = bow_vectorizer.get_feature_names_out()
df_bow_vectorized = pd.DataFrame(bow_matrix.toarray(), columns=feature_names_bow)

#  Split Data

In [80]:
X_tf = df_tfidf
y_tf = df_tf['label']

X_bow = df_bow_vectorized
y_bow = df_bow['label']

X_tf_train, X_tf_test, y_tf_train, y_tf_test = train_test_split(X_tf, y_tf, test_size = 0.2, random_state = 42)

X_bow_train, X_bow_test, y_bow_train, y_bow_test = train_test_split(X_bow, y_bow, test_size = 0.2, random_state = 42)

# Experiments

## Decision Tree - Tf-Idf

In [81]:
DT1 = DecisionTreeClassifier()
DT1.fit(X_tf_train, y_tf_train)
DT1_pred = DT1.predict(X_tf_test)

DT1_accuracy = accuracy_score(y_tf_test, DT1_pred)

print(f"Accuracy: {DT1_accuracy * 100:.2f}% \n")
print(classification_report(y_tf_test, DT1_pred))

Accuracy: 70.59% 

              precision    recall  f1-score   support

    negative       0.61      0.89      0.72        19
     neutral       0.50      0.27      0.35        11
    positive       0.94      0.76      0.84        21

    accuracy                           0.71        51
   macro avg       0.68      0.64      0.64        51
weighted avg       0.72      0.71      0.69        51



## Decision Tree - Bag of Words

In [82]:
DT2 = DecisionTreeClassifier()
DT2.fit(X_bow_train, y_bow_train)
DT2_pred = DT2.predict(X_bow_test)
DT2_accuracy = accuracy_score(y_bow_test, DT2_pred)

print(f"Accuracy: {DT2_accuracy * 100:.2f}% \n")
print(classification_report(y_bow_test, DT2_pred))

Accuracy: 62.75% 

              precision    recall  f1-score   support

    negative       0.59      0.84      0.70        19
     neutral       0.33      0.09      0.14        11
    positive       0.71      0.71      0.71        21

    accuracy                           0.63        51
   macro avg       0.55      0.55      0.52        51
weighted avg       0.59      0.63      0.58        51



## Logistic Regression - Tf-Idf

In [83]:
LR1 = LogisticRegression(max_iter=1000)
LR1.fit(X_tf_train, y_tf_train)
LR1_pred = LR1.predict(X_tf_test)

LR1_accuracy = accuracy_score(y_tf_test, LR1_pred)

print(f"Accuracy: {LR1_accuracy * 100:.2f}% \n")
print(classification_report(y_tf_test, LR1_pred))

Accuracy: 64.71% 

              precision    recall  f1-score   support

    negative       0.56      1.00      0.72        19
     neutral       0.00      0.00      0.00        11
    positive       0.88      0.67      0.76        21

    accuracy                           0.65        51
   macro avg       0.48      0.56      0.49        51
weighted avg       0.57      0.65      0.58        51



## Logistic Regression - Bag of Words

In [84]:
LR2 = LogisticRegression(max_iter=1000)
LR2.fit(X_bow_train, y_bow_train)
LR2_pred = LR2.predict(X_bow_test)

LR2_accuracy = accuracy_score(y_bow_test, LR2_pred)

print(f"Accuracy: {LR2_accuracy * 100:.2f}% \n")
print(classification_report(y_bow_test, LR2_pred))

Accuracy: 66.67% 

              precision    recall  f1-score   support

    negative       0.65      0.68      0.67        19
     neutral       0.50      0.18      0.27        11
    positive       0.70      0.90      0.79        21

    accuracy                           0.67        51
   macro avg       0.62      0.59      0.58        51
weighted avg       0.64      0.67      0.63        51

